# 04 — Match McMaster

| | |
|---|---|
| **Pipeline stage** | `match_mcmaster` |
| **Service module** | `backend.services.pipeline.match_parts_only` → `matcher.match_parts` |
| **Website equivalent** | `match_mcmaster` events in `POST /api/import/stream` |
| **Prev / Next** | `03_parse_bom.ipynb` → `05_api_payload.ipynb` |

Calls `pipeline.match_parts_only` only — same as the website **before** `enrich_mcmaster`. Live SKU/pricing hydration runs in `05_api_payload.ipynb` / `import_from_url`.


In [ ]:
from backend.models.part import Part

from backend.notebook_utils import (
    load_cached_parts,
    notebook_progress,
    parts_to_dataframe,
    prepare_crawl_env,
    resolve_parts_offline,
)
from backend.services.pipeline import match_parts_only
from backend.services.parsers.helpers.specification_checks import (
    check_parts_specifications,
    format_specification_issues,
)

prepare_crawl_env(reload_backend=False)
progress = notebook_progress("04")

parts = load_cached_parts()
if not parts:
    parts, _ = await resolve_parts_offline()

if not parts:
    from backend.services.vendors.mcmaster.cross_test import load_query_accuracy_cases

    print("Offline demo — parts from tests/fixtures/bom_listing_query_cases.json")
    parts = [case.part for case in load_query_accuracy_cases()[:5]]

spec_issues = check_parts_specifications(parts)
if spec_issues:
    print(format_specification_issues(spec_issues))

matched = match_parts_only(parts, on_progress=progress)
parts_to_dataframe(matched)
